# Tarea 1 – Parte 2: Del LLM al clasificador

**Pregunta guía:** ¿Pueden convertir titulares peruanos en un clasificador de noticias?

Datos de entrada: `datos/noticias.csv` (399 noticias de El Comercio, generado en `01_recoleccion.ipynb`).

| Paso | Contenido |
|---|---|
| 1 | Línea base zero-shot (sin entrenamiento) |
| 2 | Supervisión débil con LLM + auditoría humana |
| 3 | Partición 70 / 15 / 15 estratificada |
| 4 | Fine-tuning de DistilBERT con HuggingFace `Trainer` |
| 5 | Evaluación final en test |
| 6 | Comparativa y decisión |

**Optimización para el equipo** (MacBook, 8 GB de RAM, 6 núcleos de CPU, sin GPU dedicada):
- Todo corre en **CPU** con 4 hilos, dejando núcleos libres para el sistema.
- Se usa **un modelo pesado a la vez**: el zero-shot se libera de memoria antes de usar el LLM,
  y el LLM se descarga de Ollama antes del fine-tuning.
- Modelos pequeños: zero-shot BERT base (no *large*), Qwen 2.5 **3B** (no 7B), DistilBERT.
- Lotes pequeños (8) y textos cortos (titulares, máximo 64 tokens) para limitar la RAM.
- Resultados intermedios guardados en disco (`etiquetas_llm.csv`) para no repetir cálculos.

**Referencia para medir aciertos:** la columna `etiqueta` de `noticias.csv`, que viene de la
sección donde el diario publicó la noticia (mapeo documentado en el notebook 1).

In [1]:
import gc
import json
import random
import time
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
pd.set_option("display.max_colwidth", None)

CATEGORIAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
# Se usa CPU a proposito: el equipo tiene 8 GB de RAM y se prioriza no saturarlo
DISPOSITIVO = "cpu"
torch.set_num_threads(4)
DATOS = Path("datos")

noticias = pd.read_csv(DATOS / "noticias.csv").fillna({"bajada": ""})
print("Dispositivo:", DISPOSITIVO, "| noticias:", noticias.shape)
noticias.etiqueta.value_counts()

Dispositivo: cpu | noticias: (399, 7)


etiqueta
politica              100
economia              100
ciencia_tecnologia    100
deportes               99
Name: count, dtype: int64

## Paso 1 – Línea base zero-shot (sin fine-tuning)

Un modelo **zero-shot** clasifica en categorías que nunca vio durante su entrenamiento.
Fue entrenado en *inferencia de lenguaje natural* (NLI): dada una frase y una hipótesis,
decide si la frase la implica. Para clasificar, se arma una hipótesis por categoría
("Esta noticia trata sobre deportes.") y gana la que el modelo considera más implicada.

- Modelo: `Recognai/bert-base-spanish-wwm-cased-xnli` (BERT en español, ~110 M parámetros;
  se prefirió frente a `xlm-roberta-large-xnli` por memoria: 8 GB de RAM).
- Muestra: **40 titulares**, 10 por categoría, elegidos al azar con semilla 42.
- Este resultado es el **piso mínimo** que el modelo entrenado debe superar.

In [2]:
from transformers import pipeline

# Nombre legible para el modelo NLI -> categoria normalizada
NOMBRES_ZS = {
    "política": "politica",
    "economía": "economia",
    "deportes": "deportes",
    "ciencia y tecnología": "ciencia_tecnologia",
}
zero_shot = pipeline(
    "zero-shot-classification",
    model="Recognai/bert-base-spanish-wwm-cased-xnli",
    device=DISPOSITIVO,
)


def clasificar_zero_shot(textos: list[str]) -> list[str]:
    salidas = zero_shot(textos, candidate_labels=list(NOMBRES_ZS),
                        hypothesis_template="Esta noticia trata sobre {}.", batch_size=8)
    return [NOMBRES_ZS[s["labels"][0]] for s in salidas]


muestra_zs = noticias.groupby("etiqueta", group_keys=False).sample(10, random_state=SEMILLA)
inicio = time.perf_counter()
muestra_zs["pred_zero_shot"] = clasificar_zero_shot(muestra_zs.titular.tolist())
ms_zero_shot = (time.perf_counter() - inicio) / len(muestra_zs) * 1000

acc_zs40 = accuracy_score(muestra_zs.etiqueta, muestra_zs.pred_zero_shot)
f1_zs40 = f1_score(muestra_zs.etiqueta, muestra_zs.pred_zero_shot, average="macro")
print(f"Zero-shot en 40 titulares -> accuracy {acc_zs40:.3f} | F1 macro {f1_zs40:.3f} | {ms_zero_shot:.0f} ms/titular")

/Users/greicyrc/.venvs/qlab-tarea1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1018 19:26:00.264000 86499 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4936.90it/s]

Zero-shot en 40 titulares -> accuracy 0.700 | F1 macro 0.679 | 102 ms/titular


In [3]:
pd.crosstab(muestra_zs.etiqueta, muestra_zs.pred_zero_shot,
            rownames=["real"], colnames=["zero-shot"]).reindex(index=CATEGORIAS, columns=CATEGORIAS, fill_value=0)

zero-shot,politica,economia,deportes,ciencia_tecnologia
real,,,,
politica,10,0,0,0
economia,4,5,1,0
deportes,1,0,9,0
ciencia_tecnologia,1,1,4,4


El modelo zero-shot ocupa ~0.5 GB de RAM. Se libera la memoria antes de cargar el
LLM; en el Paso 6 se vuelve a cargar para evaluarlo sobre el test.

In [4]:
del zero_shot
gc.collect()
print("Modelo zero-shot liberado de memoria")

Modelo zero-shot liberado de memoria


## Paso 2 – Supervisión débil con LLM

Etiquetar 399 noticias a mano es lento; un LLM lo hace en minutos, pero **puede equivocarse**.
Por eso se llama supervisión *débil*: son etiquetas útiles para entrenar, no una verdad absoluta.
La auditoría humana mide cuánto podemos confiar en ellas.

- LLM: **Qwen 2.5 3B** ejecutado localmente con **Ollama** (gratis, sin claves de API, los datos no salen del equipo).
  Se usó la versión de 3B porque la de 7B no cabe con holgura en 8 GB de RAM.
- Entrada: titular **y** bajada (la bajada aclara titulares ambiguos, como columnas de opinión).
- Salida forzada a **JSON con esquema**: Ollama solo permite responder una de las 4 categorías y
  un número de confianza, así no hay que interpretar texto libre.
- `temperature=0`: la misma noticia siempre recibe la misma respuesta (reproducible).
- `num_gpu=0`, `num_thread=4`: el modelo corre **solo en CPU** con 4 hilos.
- `num_ctx=1024`: ventana de contexto corta (el prompt + noticia ocupa ~300 tokens);
  el valor por defecto reserva mucha más RAM sin necesidad.

In [5]:
import requests

OLLAMA = "http://localhost:11434/api/chat"
MODELO_LLM = "qwen2.5:3b"

PROMPT_SISTEMA = """Eres un editor de un diario peruano. Clasifica cada noticia en UNA de estas 4 categorías:
- politica: Congreso, Gobierno, Poder Judicial, elecciones, partidos, política internacional.
- economia: mercados, empresas, empleo, precios, impuestos, presupuesto, finanzas.
- deportes: fútbol y cualquier otra disciplina deportiva, deportistas, torneos.
- ciencia_tecnologia: ciencia, salud científica, medio ambiente, tecnología, internet, IA, videojuegos, gadgets.
Responde solo con JSON: {"categoria": <una de las 4>, "confianza": <número entre 0 y 1>}."""

ESQUEMA = {
    "type": "object",
    "properties": {
        "categoria": {"type": "string", "enum": CATEGORIAS},
        "confianza": {"type": "number"},
    },
    "required": ["categoria", "confianza"],
}


def etiquetar_llm(titular: str, bajada: str) -> dict:
    mensaje = f"Titular: {titular}\nBajada: {bajada or '(sin bajada)'}"
    r = requests.post(OLLAMA, timeout=120, json={
        "model": MODELO_LLM,
        "messages": [{"role": "system", "content": PROMPT_SISTEMA},
                     {"role": "user", "content": mensaje}],
        "format": ESQUEMA,
        "stream": False,
        "options": {"temperature": 0, "seed": SEMILLA, "num_gpu": 0, "num_thread": 4, "num_ctx": 1024},
        "keep_alive": "10m",  # mantiene el modelo cargado solo mientras se etiqueta
    })
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])


# Prueba con una noticia antes de lanzar las 399
fila = noticias.iloc[0]
print(fila.titular, "->", etiquetar_llm(fila.titular, fila.bajada))

El voto 31: Este es el escenario que muestra el peso de Obras para el fujimorismo en la delegación de facultades -> {'categoria': 'politica', 'confianza': 0.9}


In [6]:
ARCHIVO_LLM = DATOS / "etiquetas_llm.csv"

# Reanudable: si el etiquetado se interrumpe, continua desde la ultima noticia guardada
hechas = pd.read_csv(ARCHIVO_LLM) if ARCHIVO_LLM.exists() else pd.DataFrame(columns=["url"])
pendientes = noticias[~noticias.url.isin(hechas.url)]
print(f"Ya etiquetadas: {len(hechas)} | pendientes: {len(pendientes)}")

filas, inicio = [], time.perf_counter()
for i, n in enumerate(pendientes.itertuples(), 1):
    try:
        res = etiquetar_llm(n.titular, n.bajada)
        etiqueta, confianza = res["categoria"], float(res["confianza"])
    except Exception as e:  # una falla no detiene todo: se registra y se sigue
        print(f"  {n.url}: error {e}")
        etiqueta, confianza = None, None
    filas.append({"titular": n.titular, "etiqueta_llm": etiqueta, "confianza": confianza,
                  "modelo_llm": MODELO_LLM, "fecha": date.today().isoformat(), "url": n.url})
    if i % 25 == 0 or i == len(pendientes):   # guardado parcial cada 25 noticias
        hechas = pd.concat([hechas, pd.DataFrame(filas)], ignore_index=True)
        hechas.to_csv(ARCHIVO_LLM, index=False, encoding="utf-8")
        filas = []
        print(f"  {len(hechas)}/{len(noticias)} etiquetadas")
if len(pendientes):
    print(f"Etiquetado en {(time.perf_counter() - inicio) / 60:.1f} min "
          f"({(time.perf_counter() - inicio) / len(pendientes):.1f} s por noticia en CPU)")

# Liberar la RAM que ocupa el LLM en Ollama antes de entrenar
requests.post(OLLAMA, json={"model": MODELO_LLM, "messages": [], "keep_alive": 0}, timeout=60)
etiquetas_llm = pd.read_csv(ARCHIVO_LLM)
etiquetas_llm.head()

Ya etiquetadas: 0 | pendientes: 399


  25/399 etiquetadas


  50/399 etiquetadas


  75/399 etiquetadas


  100/399 etiquetadas


...

  325/399 etiquetadas


  350/399 etiquetadas


  375/399 etiquetadas


  399/399 etiquetadas
Etiquetado en 50.3 min (7.6 s por noticia en CPU)


,url,titular,etiqueta_llm,confianza,modelo_llm,fecha
0,https://elcomercio.pe/politica/el-voto-31-este-es-el-escenario-que-muestra-el-peso-de-obras-para-el-fujimorismo-en-la-delegacion-de-facultades-congreso-keiko-fujimori-senado-apuesplan-noticia/,El voto 31: Este es el escenario que muestra el peso de Obras para el fujimorismo en la delegación de facultades,politica,0.9,qwen2.5:3b,2026-10-18
1,https://elcomercio.pe/politica/gobierno/jorge-nieto-dice-que-se-reunio-con-el-ejecutivo-aprobemos-los-20-puntos-y-hagannos-llegar-como-proyecto-de-ley-los-otros-7-puntos-que-consideran-imprescindibles-ultimas-apuesplan-noticia/,Jorge Nieto dice que se reunió con el Ejecutivo: “Aprobemos los 20 puntos y hágannos llegar como proyecto de ley los otros 7”,politica,0.9,qwen2.5:3b,2026-10-18
2,https://elcomercio.pe/politica/alvaro-vargas-llosa-eeuu-embajador-del-peru-en-estados-unidos-entrevista-noticia/,Álvaro Vargas Llosa: “EE.UU. nos está ayudando a recuperar la soberanía que las mafias ponen en peligro”,politica,0.9,qwen2.5:3b,2026-10-18
3,https://elcomercio.pe/politica/el-senado-tambien-se-la-juega-una-cronica-de-fernando-vivas-congreso-facultades-legislativas-keiko-fujimori-gobierno-apuesplan-noticia/,"El Senado también se la juega, una crónica de Fernando Vivas sobre la pugna por las facultades legislativas",politica,0.9,qwen2.5:3b,2026-10-18
4,https://elcomercio.pe/politica/comision-de-constitucion-del-senado-reanuda-este-lunes-debate-sobre-facultades-al-ejecutivo-en-seguridad-y-fen-apuesplan-noticia/,Comisión de Constitución del Senado reanuda este lunes debate sobre el pedido de facultades legislativas,politica,0.9,qwen2.5:3b,2026-10-18


In [7]:
df = noticias.merge(etiquetas_llm[["url", "etiqueta_llm", "confianza"]], on="url", how="left")
print("Sin etiqueta valida del LLM:", df.etiqueta_llm.isna().sum())
acc_llm_total = accuracy_score(df.etiqueta, df.etiqueta_llm.fillna("sin_etiqueta"))
print(f"Coincidencia LLM vs seccion del diario (399 noticias): {acc_llm_total:.1%}")
pd.crosstab(df.etiqueta, df.etiqueta_llm, rownames=["seccion"], colnames=["LLM"]).reindex(index=CATEGORIAS, columns=CATEGORIAS, fill_value=0)

Sin etiqueta valida del LLM: 3
Coincidencia LLM vs seccion del diario (399 noticias): 89.0%


LLM,politica,economia,deportes,ciencia_tecnologia
seccion,,,,
politica,100,0,0,0
economia,16,77,0,7
deportes,2,0,94,0
ciencia_tecnologia,8,7,1,84


### 2.1 Auditoría humana (20 casos al azar)

Se eligen 20 noticias al azar (semilla 42) y **una persona** decide si la etiqueta del LLM es
correcta leyendo titular y bajada. La revisión se registra en `datos/auditoria_humana.csv`,
columna `acuerdo_humano` (1 = de acuerdo, 0 = en desacuerdo) y `comentario` cuando no hay acuerdo.

> Esta columna la completa la estudiante a mano; no la genera el código ni una IA.

In [8]:
ARCHIVO_AUDITORIA = DATOS / "auditoria_humana.csv"

if not ARCHIVO_AUDITORIA.exists():
    auditoria = df.sample(20, random_state=SEMILLA)[["url", "titular", "bajada", "etiqueta_llm", "confianza"]]
    auditoria["acuerdo_humano"] = ""
    auditoria["comentario"] = ""
    auditoria.to_csv(ARCHIVO_AUDITORIA, index=False, encoding="utf-8")
    print("Plantilla creada: completar la columna acuerdo_humano y volver a ejecutar.")

auditoria = pd.read_csv(ARCHIVO_AUDITORIA)
auditoria[["titular", "bajada", "etiqueta_llm", "acuerdo_humano", "comentario"]]

Plantilla creada: completar la columna acuerdo_humano y volver a ejecutar.


,titular,bajada,etiqueta_llm,acuerdo_humano,comentario
0,Precio del dólar del lunes 12 de octubre: a cuánto cerró el tipo de cambio,Consulta la cotización en compra y venta de la moneda estadounidense en el país de este lunes 12 de octubre.,economia,NaN,NaN
1,"Reconocimiento facial, análisis de redes y rastreo digital: así vigilan hoy una protesta de ‘Gen Z’","La tecnología se ha vuelto un factor clave en India, durante las manifestaciones de las “Cucarachas”. Incluso se recurre a la inteligencia artificial y vehículos de vigilancia denominados Ikshana",ciencia_tecnologia,NaN,NaN
2,Municipalidad de Lima y JNE discrepan sobre los alcances del fallo por las demoras electorales,"La comuna de Lima sostiene que la sentencia reconoce afectaciones al derecho al voto y exige responsabilidades, mientras que el organismo electoral precisa que no se ordenaron nuevas elecciones y que las dispensas de multas deben tramitarse conforme a la normativa vigente.",politica,NaN,NaN
3,¿Dónde ver Universitario vs. Alianza Atlético EN VIVO por fecha 12 del Torneo Clausura de la Liga 1?,Revisa qué canales pasan el partido de Universitario vs. Alianza Atlético y los horarios para ver el compromiso por la fecha 12 del Torneo Clausura de la Liga 1.,deportes,NaN,NaN
4,PPK: Poder Judicial levanta impedimento de salida del país dictada contra el expresidente,"Cuarto Juzgado de Investigación Preparatoria Nacional también declaró ejecutoriada la decisión de la Tercera Sala Penal de Apelaciones Nacional, que archivó el proceso.",politica,NaN,NaN
5,"Giannina, la impasible, una crónica de Fernando Vivas sobre la presidenta de la Comisión de Constitución de Diputados","La presidenta de la Comisión de Constitución, Giannina Avendaño, de Juntos Por el Perú, sacó de quicio al fujimorismo con su impasible rechazo a aprobar las facultades.",politica,NaN,NaN
6,Ciberamenazas en América Latina: Perú registró mayor actividad de malware en el primer semestre de 2026,ESET Latinoamérica revela que los ataques no necesitaron técnicas innovadoras. El programa malicioso Win/PSW.Amatera es uno de los más recurrentes en el país para el robo de información,ciencia_tecnologia,NaN,NaN
7,Daniel Barragán sobre frase usada por Keiko Fujimori para criticar a la oposición: “No ayuda a poner paños fríos”,"El senador de Obras cuestionó la expresión de la presidenta, comúnmente atribuida al Quijote, y pidió al Congreso y al Ejecutivo actuar con responsabilidad y madurez.",politica,NaN,NaN
8,"¿Dónde ver repetición del partido, Sporting Cristal 4-2 Comerciantes por Torneo Clausura, Liga 1?",Sporting Cristal derrotó 4-2 a Comerciantes Unidos con dobletes de Hernán Barcos y Luis Iberico por la fecha 12 del Torneo Clausura de la Liga 1 en el estadio Alberto Gallardo.,deportes,NaN,NaN
9,MTC plantea reducir la TUUA de transferencia internacional en “no menos del 50%” en el aeropuerto Jorge Chávez,"El ministro de Transportes y Comunicaciones (MTC), José Angello Tangherlini, señaló que la elaboración del borrador de la propuesta estaría avanzado al final de la próxima semaan. El Gobierno también busca que el cobro se incorpore directamente en los pasajes aéreos.",politica,NaN,NaN


In [9]:
revisadas = pd.to_numeric(auditoria.acuerdo_humano, errors="coerce")
if revisadas.notna().sum() < len(auditoria):
    print(f"Auditoria incompleta: {revisadas.notna().sum()} de {len(auditoria)} revisadas.")
    acuerdo_humano = None
else:
    aciertos = int(revisadas.sum())
    acuerdo_humano = aciertos / len(auditoria)
    print(f"Acuerdo LLM-humano: {aciertos}/{len(auditoria)} = {acuerdo_humano:.0%}")

Auditoria incompleta: 0 de 20 revisadas.


## Paso 3 – Partición de datos (70 / 15 / 15)

- Se entrena con las **etiquetas del LLM** (supervisión débil); las noticias sin etiqueta válida se excluyen.
- Split **estratificado** por `etiqueta_llm`: cada parte conserva la proporción de categorías.
- `random_state=42` en ambos cortes: cualquiera que ejecute el notebook obtiene la misma partición.
- El **test queda guardado** y no se usa hasta el Paso 5 (ni para elegir época ni para ajustar nada).
- La partición se guarda en `datos/particion.csv` para poder auditarla.

In [ ]:
from sklearn.model_selection import train_test_split

datos_modelo = df.dropna(subset=["etiqueta_llm"]).reset_index(drop=True)
train, temp = train_test_split(datos_modelo, test_size=0.30, stratify=datos_modelo.etiqueta_llm, random_state=SEMILLA)
val, test = train_test_split(temp, test_size=0.50, stratify=temp.etiqueta_llm, random_state=SEMILLA)

particion = pd.concat([train.assign(split="train"), val.assign(split="val"), test.assign(split="test")])
particion[["url", "split"]].to_csv(DATOS / "particion.csv", index=False)

resumen = pd.crosstab(particion.etiqueta_llm, particion.split)[["train", "val", "test"]]
resumen.loc["TOTAL"] = resumen.sum()
print({k: f"{len(v)} ({len(v) / len(datos_modelo):.0%})" for k, v in [("train", train), ("val", val), ("test", test)]})
resumen

## Paso 4 – Fine-tuning de DistilBERT

**Fine-tuning** = tomar un modelo que ya "sabe" español (pre-entrenado en millones de textos)
y ajustarlo con nuestros ejemplos para esta tarea concreta.

- Modelo: `distilbert-base-multilingual-cased` (versión reducida de BERT, ~135 M parámetros).
- Entrada: **solo el titular**, porque el dashboard clasificará titulares nuevos.
- HuggingFace `Trainer`: 3 épocas, lote de 8, evaluación en validación al final de cada época,
  y se conserva la época con **mejor F1 macro en validación** (`load_best_model_at_end`).
- Optimizado para CPU / 8 GB: lote de 8, máximo 64 tokens, un solo checkpoint en disco.

In [ ]:
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          Trainer, TrainingArguments)

MODELO_BASE = "distilbert-base-multilingual-cased"
DIR_MODELO = Path("modelo")
id2label = dict(enumerate(CATEGORIAS))
label2id = {c: i for i, c in id2label.items()}

tokenizer = AutoTokenizer.from_pretrained(MODELO_BASE)


def a_dataset(parte: pd.DataFrame) -> Dataset:
    ds = Dataset.from_dict({"text": parte.titular.tolist(),
                            "label": parte.etiqueta_llm.map(label2id).tolist()})
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=64), batched=True)


ds_train, ds_val, ds_test = a_dataset(train), a_dataset(val), a_dataset(test)
print(ds_train)

In [ ]:
def metricas(pred):
    y_pred = pred.predictions.argmax(-1)
    return {"accuracy": accuracy_score(pred.label_ids, y_pred),
            "f1_macro": f1_score(pred.label_ids, y_pred, average="macro")}


modelo = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BASE, num_labels=len(CATEGORIAS), id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir=str(DIR_MODELO),          # checkpoints en modelo/checkpoint-* (ignorados por git)
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,                  # un solo checkpoint en disco
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_strategy="epoch",
    use_cpu=True,
    dataloader_num_workers=0,
    seed=SEMILLA,
    report_to="none",
)
trainer = Trainer(model=modelo, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                  processing_class=tokenizer, compute_metrics=metricas)

inicio = time.perf_counter()
trainer.train()
min_entrenamiento = (time.perf_counter() - inicio) / 60
print(f"Entrenamiento: {min_entrenamiento:.1f} min en CPU")

In [ ]:
historial = pd.DataFrame([h for h in trainer.state.log_history if "eval_f1_macro" in h])
historial[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]].round(3)

In [ ]:
trainer.save_model(DIR_MODELO)   # model.safetensors (~540 MB, fuera de git) + config + tokenizer
import shutil

for ckpt in DIR_MODELO.glob("checkpoint-*"):   # ya no se necesitan: el mejor modelo esta guardado
    shutil.rmtree(ckpt)
sorted(p.name for p in DIR_MODELO.iterdir())

## Paso 5 – Evaluación final en test

Primera y única vez que el modelo ve el test. Se compara contra la **sección del diario**
(referencia independiente del LLM) y, como dato adicional, contra las etiquetas del LLM.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

inicio = time.perf_counter()
pred_test = trainer.predict(ds_test)
ms_distilbert = (time.perf_counter() - inicio) / len(test) * 1000
test = test.assign(pred_distilbert=[id2label[i] for i in pred_test.predictions.argmax(-1)])

acc_ft = accuracy_score(test.etiqueta, test.pred_distilbert)
f1_ft = f1_score(test.etiqueta, test.pred_distilbert, average="macro")
f1_ft_vs_llm = f1_score(test.etiqueta_llm, test.pred_distilbert, average="macro")
print(f"DistilBERT en test ({len(test)} noticias) vs seccion -> accuracy {acc_ft:.3f} | F1 macro {f1_ft:.3f}")
print(f"DistilBERT en test vs etiquetas LLM -> F1 macro {f1_ft_vs_llm:.3f}")
print(classification_report(test.etiqueta, test.pred_distilbert, labels=CATEGORIAS, digits=3, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

mc = confusion_matrix(test.etiqueta, test.pred_distilbert, labels=CATEGORIAS)
fig, ax = plt.subplots(figsize=(6.5, 5))
sns.heatmap(mc, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CATEGORIAS, yticklabels=CATEGORIAS, ax=ax)
ax.set(xlabel="Predicción DistilBERT", ylabel="Real (sección del diario)",
       title=f"Matriz de confusión en test (n={len(test)}) – F1 macro {f1_ft:.2f}")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
errores = test[test.etiqueta != test.pred_distilbert][["titular", "etiqueta", "etiqueta_llm", "pred_distilbert"]]
print(f"Errores en test: {len(errores)} de {len(test)}")
errores

## Paso 6 – Comparativa y decisión

Los tres enfoques se miden en **el mismo test** y contra la **misma referencia** (sección del diario),
para que la comparación sea justa. El zero-shot se vuelve a cargar para clasificar el test completo.

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="Recognai/bert-base-spanish-wwm-cased-xnli", device=DISPOSITIVO)
inicio = time.perf_counter()
test["pred_zero_shot"] = clasificar_zero_shot(test.titular.tolist())
ms_zero_shot_test = (time.perf_counter() - inicio) / len(test) * 1000
del zero_shot
gc.collect()

# Latencia del LLM: se mide con 10 noticias del test (el etiquetado completo pudo venir del CSV)
inicio = time.perf_counter()
for n in test.head(10).itertuples():
    etiquetar_llm(n.titular, n.bajada)
ms_llm = (time.perf_counter() - inicio) / 10 * 1000
requests.post(OLLAMA, json={"model": MODELO_LLM, "messages": [], "keep_alive": 0}, timeout=60)


def fila(nombre, pred, ms, costo):
    return {"modelo": nombre,
            "accuracy": accuracy_score(test.etiqueta, pred),
            "f1_macro": f1_score(test.etiqueta, pred, average="macro"),
            "ms_por_noticia": round(ms), "costo / requisitos": costo}


comparativa = pd.DataFrame([
    fila("Zero-shot (BERT español XNLI)", test.pred_zero_shot, ms_zero_shot_test,
         "sin entrenamiento; ~0.5 GB RAM"),
    fila("LLM débil (Qwen 2.5 3B, Ollama)", test.etiqueta_llm, ms_llm,
         "sin entrenamiento; ~2.5 GB RAM; usa titular + bajada"),
    fila("DistilBERT fine-tuned", test.pred_distilbert, ms_distilbert,
         f"entrenamiento {min_entrenamiento:.0f} min en CPU; ~0.6 GB RAM"),
]).round(3)
comparativa

In [ ]:
metricas_finales = {
    "n_noticias": int(len(noticias)),
    "n_test": int(len(test)),
    "zero_shot_40": {"accuracy": round(acc_zs40, 3), "f1_macro": round(f1_zs40, 3)},
    "comparativa_test": comparativa.drop(columns="costo / requisitos").to_dict(orient="records"),
    "distilbert_test": {"accuracy": round(acc_ft, 3), "f1_macro": round(f1_ft, 3),
                        "f1_macro_vs_llm": round(f1_ft_vs_llm, 3)},
    "acuerdo_llm_humano": None if acuerdo_humano is None else round(acuerdo_humano, 3),
    "auditoria_n": int(len(auditoria)),
    "modelo_llm": MODELO_LLM,
    "modelo_base": MODELO_BASE,
}
(DIR_MODELO / "metricas.json").write_text(json.dumps(metricas_finales, indent=2, ensure_ascii=False))
print(json.dumps(metricas_finales, indent=2, ensure_ascii=False))